# Auto-Label IMU Gesture Data for MPLAB ML Development Suite

This notebook automatically labels IMU gesture data from your device with button trigger for use in MPLAB ML Development Suite.

## About This Notebook

Gesture based Machine learning models require a ton of data to train them. After acquiring such data, labeling them manually would be tedious, laborious and extremely time-consuming. This notebook allows you to automatically label the data using a trigger signal that can be raised at the start of the gesture which remains high for the duration of the gesture. An example of such an implementation would be to press a button and keep it pressed while performing the gesture and immediately releasing it on completion.
This notebook marks the data collected with the desired label for the portions the trigger signal is high.


### Folder Setup

1.	Create a folder with the ML Development Suite project name.
2.	Inside this folder, create as many sub-folders as the labels with each folder name following the format: LABEL-gesture or LABEL_gesture.
3.	Perform each gesture as  many times as required (at least 10 samples)  and save them in individual CSV files. Name the CSV files in the format LABEL_N where N is an incremental number. These CSV files will be used to train the gesture. Similarly, collect data in CSV files for testing the model. Name the CSV files in the format LABEL_tN where N is an incremental number.
```

PROJECT_NAME/
├── labelA-gesture/
│   ├── labelA_1.csv
│   ├── labelA_2.csv
│   └── labelA_t1.csv
├── labelB-gesture/
│   └── labelB_1.csv
│   └── labelB_2.csv
│   └── labelB_t1.csv
└── labelC-gesture/
    └── labelC.csv
```


### Workflow of this Notebook

1. **Upload Data** - Your recorded gesture CSV files
2. **Detect Events** - Find button press/release (rising/falling edges)
3. **Auto-Label** - Assign labels to gesture segments baseed on folder names
3. **Create Metadata** - Assign metadata - Test/Train to captues based on filename
4. **Create Project** - Generate MPLAB ML-compatible files (.dclproj)
5. **Upload to MPLAB** - Train your model in MPLAB ML

---

## Table of Contents

**Section 1**: Setup and Data Upload

**Section 2**: Gesture Segmentation and Labeling

**Section 3**: MPLAB ML Project Creation

**Section 4**: Upload to MPLAB ML

---

## Section 1: Setup and Data Upload

### 1.1 Import Required Libraries

We'll use standard Python libraries for data processing and the MPLAB ML SDK for project creation. The MPLAB ML SDK now supports Python 3.12, so we can use Colab's default Python environment.

In [1]:
# Standard library imports
import os
import glob
import json
import sys
from pathlib import Path

# Data processing
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

# Display versions
print("Environment Information:")
print(f"  Python: {sys.version.split()[0]}")
print(f"  Pandas: {pd.__version__}")
print(f"  NumPy: {np.__version__}")
print("\n✓ Libraries imported successfully")

Environment Information:
  Python: 3.12.12
  Pandas: 2.2.2
  NumPy: 2.0.2

✓ Libraries imported successfully


### 1.2 Upload Your Gesture Data

#### Data Organization

The root folder should be named after your project name. Example - MagicWand3Gesture. Your data should be organized in folders by gesture type. Each gesture should have its own folder with CSV files containing the recorded sensor data. Gesture folder name should be of the format- <gesture>-gesture. Example - horizontal-gesture, etc. Train data filename should end with _N and test data filename should end with _tN (Where N is the file number).

**Expected folder structure:**
```
MagicWand3Gesture/
├── horizontal-gesture/
│   ├── horizontal_1.csv
│   ├── horizontal_2.csv
│   └── horizontal_t1.csv
├── vertical-gesture/
│   └── vertical_1.csv
│   └── vertical_2.csv
│   └── vertical_t1.csv
├── round-gesture/
│   └── round_1.csv
│   └── round_2.csv
│   └── round_t1.csv
└── idle-gesture/
    └── idle1.csv

```

#### CSV File Format

Each CSV file should contain these columns:
- `AccelerometerX` - X-axis acceleration
- `AccelerometerY` - Y-axis acceleration
- `AccelerometerZ` - Z-axis acceleration
- `GyroscopeX` - X-axis angular velocity
- `GyroscopeY` - Y-axis angular velocity
- `GyroscopeZ` - Z-axis angular velocity
- `Trigger` - Button state (1 = pressed, 0 = released)

#### How to Upload

1. Compress your `project` folder containing training data to a ZIP file on your computer
2. Upload the ZIP file to the /content folder


### 1.3 Extract Your Gesture Data

In [2]:

import zipfile

# List available ZIP files
zip_files = glob.glob('/content/*.zip')

if zip_files:
    print(f"Found {len(zip_files)} ZIP file(s):")
    for zf in zip_files:
        print(f"  - {os.path.basename(zf)}")

    # Extract the first ZIP file found
    zip_path = zip_files[0]
    print(f"\nExtracting: {os.path.basename(zip_path)}")

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        names = zip_ref.namelist()
        zip_ref.extractall('/content/')

    print("✓ Files extracted to /content/")
    # Detect top-level folder inside ZIP
    top_folders = set(name.split('/')[0] for name in names if '/' in name)

    if len(top_folders) == 1:
        # Normal case: ZIP contains one root folder
        folder_name = list(top_folders)[0]
        DATA_DIR = os.path.join('/content', folder_name)
    else:
        # ZIP contains loose files
        DATA_DIR = '/content'

    print("Dataset path detected:")
    print("Folder name:", folder_name)
    print(DATA_DIR)

else:
    print("No ZIP files found in /content/")
    print("If you uploaded individual files, you can skip this cell.")

Found 1 ZIP file(s):
  - MagicWandSimple.zip

Extracting: MagicWandSimple.zip
✓ Files extracted to /content/
Dataset path detected:
Folder name: MagicWandSimple
/content/MagicWandSimple


### 1.4 Verify Data Upload

Let's check that your data was uploaded correctly and is in the expected location.

In [3]:
# List all subdirectories (gesture folders)
gesture_folders = [d for d in os.listdir(DATA_DIR)
                      if os.path.isdir(os.path.join(DATA_DIR, d))]

print(f"Found {len(gesture_folders)} gesture folder(s):")

total_files = 0
for folder in sorted(gesture_folders):
    folder_path = os.path.join(DATA_DIR, folder)
    csv_files = glob.glob(os.path.join(folder_path, '*.csv'))
    total_files += len(csv_files)
    print(f"\n  📁 {folder}/")
    for csv_file in sorted(csv_files):
        file_size = os.path.getsize(csv_file) / 1024  # KB
        print(f"     - {os.path.basename(csv_file):30s} ({file_size:>8.1f} KB)")

print(f"\n{'='*60}")
print(f"Total: {total_files} CSV file(s) ready for processing")
print(f"{'='*60}")



Found 4 gesture folder(s):

  📁 horizontal-gesture/
     - horizontal_1.csv               (    99.5 KB)
     - horizontal_10.csv              (   119.6 KB)
     - horizontal_11.csv              (   181.6 KB)
     - horizontal_12.csv              (    85.8 KB)
     - horizontal_13.csv              (   191.8 KB)
     - horizontal_2.csv               (   183.9 KB)
     - horizontal_3.csv               (   137.8 KB)
     - horizontal_4.csv               (   119.9 KB)
     - horizontal_5.csv               (   110.7 KB)
     - horizontal_6.csv               (   206.5 KB)
     - horizontal_7.csv               (   120.0 KB)
     - horizontal_8.csv               (   112.7 KB)
     - horizontal_9.csv               (   122.8 KB)
     - horizontal_t1.csv              (    30.2 KB)
     - horizontal_t10.csv             (    71.0 KB)
     - horizontal_t11.csv             (    48.5 KB)
     - horizontal_t12.csv             (   259.3 KB)
     - horizontal_t2.csv              (    46.7 KB)
     - horiz

In [4]:
# Find all CSV files
all_csv_files = []
for gesture_folder in sorted(os.listdir(DATA_DIR)):
    folder_path = os.path.join(DATA_DIR, gesture_folder)
    if os.path.isdir(folder_path):
        csv_files = glob.glob(os.path.join(folder_path, '*.csv'))
        all_csv_files.extend(csv_files)

if not all_csv_files:
    print("No CSV files found!")
else:
    # Load the first CSV file as a sample
    sample_file = all_csv_files[0]
    df_sample = pd.read_csv(sample_file)

    print(f"Sample file: {os.path.basename(sample_file)}")
    print(f"Full path: {sample_file}")
    print(f"\nDataset shape: {df_sample.shape[0]:,} rows × {df_sample.shape[1]} columns")
    print(f"\nColumn names:")
    for i, col in enumerate(df_sample.columns, 1):
        print(f"  {i}. {col}")

    print(f"\nData types:")
    print(df_sample.dtypes)

    print(f"\nFirst 10 rows:")
    print(df_sample.head(10))

Sample file: horizontal_8.csv
Full path: /content/MagicWandSimple/horizontal-gesture/horizontal_8.csv

Dataset shape: 4,107 rows × 7 columns

Column names:
  1. AccelerometerX
  2. AccelerometerY
  3. AccelerometerZ
  4. GyroscopeX
  5. GyroscopeY
  6. GyroscopeZ
  7. Trigger

Data types:
AccelerometerX    int64
AccelerometerY    int64
AccelerometerZ    int64
GyroscopeX        int64
GyroscopeY        int64
GyroscopeZ        int64
Trigger           int64
dtype: object

First 10 rows:
   AccelerometerX  AccelerometerY  AccelerometerZ  GyroscopeX  GyroscopeY  \
0            2058            -206             -25          23          24   
1            2065            -216             -61          -6         -32   
2            2066            -244             -89         -44           5   
3            2056            -265             -92         -79          73   
4            2089            -243             -84        -111         112   
5            2093            -222             -62 

## Section 2: Gesture Segmentation and Labeling

Now we'll automatically label your gesture data based on button trigger signals. Each segment between a button press (rising edge) and release (falling edge) will be labeled according to the gesture name extracted from the filename.

### 2.1 Define Gesture Labels

First, let's define the mapping between gesture names and numeric label IDs for MPLAB ML. The sub-folder name is used to generate the labels.

In [5]:
import re

print("Scanning folders inside:", DATA_DIR)

# Get only valid subfolders
subfolders = [
    f for f in os.listdir(DATA_DIR)
    if os.path.isdir(os.path.join(DATA_DIR, f))
    and not f.startswith('.')  # ignore hidden folders
]

print("Found folders:", subfolders)

# Extract label names (split at '-' OR '_')
labels = []

for folder in subfolders:
    # Split at first '-' or '_'
    label = re.split(r'[-_]', folder)[0]
    labels.append(label)

# Remove duplicates + sort for stable ordering
labels = sorted(set(labels))

print("Extracted labels:", labels)

# Create label mapping
LABEL_MAP = {label: idx for idx, label in enumerate(labels)}

# Optional unknown class
LABEL_MAP['dontcare'] = len(LABEL_MAP)

# Reverse mapping
LABEL_NAMES = {v: k for k, v in LABEL_MAP.items()}

# Print mapping
print("\nGesture Label Mapping:")
print("="*40)
for gesture, label_id in LABEL_MAP.items():
    print(f"  {gesture:10s} -> {label_id}")

Scanning folders inside: /content/MagicWandSimple
Found folders: ['vertical-gesture', 'horizontal-gesture', 'idle-gesture', 'round-gesture']
Extracted labels: ['horizontal', 'idle', 'round', 'vertical']

Gesture Label Mapping:
  horizontal -> 0
  idle       -> 1
  round      -> 2
  vertical   -> 3
  dontcare   -> 4


### 2.2 Labeling Functions

These functions will:
1. Extract gesture name from filename (e.g., "horizontal_1.csv" -> "horizontal")
2. Detect button press/release edges
3. Label segments between edges

In [6]:
def extract_gesture_name(filename):
    """
    Extract gesture name from filename.
    Examples: 'horizontal_1.csv' -> 'horizontal', 'horizontal-gesture/horizontal_t1.csv' -> 'horizontal'
    """
    basename = os.path.basename(filename)
    name = os.path.splitext(basename)[0]

    # Extract first part before '_'
    if '_' in name:
        gesture = name.split('_')[0]
    else:
        gesture = name.split('-')[0]

    return gesture.lower()


def detect_button_edges(trigger_signal):
    """
    Detect rising and falling edges in trigger signal.

    Returns:
        rising_edges: indices where button was pressed (0->1)
        falling_edges: indices where button was released (1->0)
    """
    trigger = np.array(trigger_signal)
    diff = np.diff(trigger, prepend=0)

    rising_edges = np.where(diff == 1)[0]
    falling_edges = np.where(diff == -1)[0]

    return rising_edges, falling_edges


def label_gesture_data(df, gesture_name, label_map=LABEL_MAP,
                      min_samples=10, max_idle_samples=100):
    """
    Label IMU data based on trigger signal edges.

    Args:
        df: DataFrame with IMU data and Trigger column
        gesture_name: Name of the gesture (e.g., 'horizontal', 'vertical')
        label_map: Dictionary mapping gesture names to label IDs
        min_samples: Minimum samples for a valid gesture segment
        max_idle_samples: Maximum samples to include for idle class

    Returns:
        DataFrame with added 'label' column
    """
    df = df.copy()

    # Initialize all samples as 'dontcare'
    df['label'] = label_map['dontcare']

    # Detect button edges
    rising_edges, falling_edges = detect_button_edges(df['Trigger'])

    gesture_count = 0

    # Label each segment
    for start_idx in rising_edges:
        # Find corresponding falling edge
        matching_falls = falling_edges[falling_edges > start_idx]

        if len(matching_falls) == 0:
            # Button pressed but not released - use end of recording
            end_idx = len(df) - 1
        else:
            end_idx = matching_falls[0]

        segment_length = end_idx - start_idx

        # Skip segments that are too short
        if segment_length < min_samples:
            continue

        # Label the segment
        if gesture_name == 'idle':
            # For idle, limit segment length
            actual_end = min(start_idx + max_idle_samples, end_idx)
            df.loc[start_idx:actual_end, 'label'] = label_map['idle']
        elif gesture_name in label_map:
            df.loc[start_idx:end_idx, 'label'] = label_map[gesture_name]

        gesture_count += 1

    return df, gesture_count


print("Labeling functions defined")

Labeling functions defined


### 2.3 Process All Gesture Files

Now we'll process each CSV file, label the segments, and save the results.

In [7]:
# Create output directory for labeled files
LABELED_DIR = '/content/labeled_data'
os.makedirs(LABELED_DIR, exist_ok=True)

# Process each CSV file
print("Processing gesture files...")
print("="*70)

labeled_files = []
total_files = 0
total_gestures = 0

for csv_file in sorted(all_csv_files):
    # Extract gesture name
    gesture = extract_gesture_name(csv_file)

    # Load data
    df = pd.read_csv(csv_file)

    # Label the data
    df_labeled, gesture_count = label_gesture_data(df, gesture)

    # Create output filename
    basename = os.path.basename(csv_file)
    output_filename = f"{gesture}_labeled_{basename}"
    output_path = os.path.join(LABELED_DIR, output_filename)

    # Save labeled data
    df_labeled.to_csv(output_path, index=False)
    labeled_files.append(output_path)

    total_files += 1
    total_gestures += gesture_count

    # Print summary
    label_counts = df_labeled['label'].value_counts().sort_index()
    print(f"\n{basename}")
    print(f"  Gesture: {gesture}")
    print(f"  Segments labeled: {gesture_count}")
    print(f"  Total samples: {len(df_labeled):,}")

    for label_id, count in label_counts.items():
        label_name = LABEL_NAMES.get(label_id, f"Unknown({label_id})")
        pct = 100.0 * count / len(df_labeled)
        print(f"    {label_name:10s}: {count:6,} samples ({pct:5.1f}%)")

print(f"\n{'='*70}")
print(f"Processing complete!")
print(f"  Files processed: {total_files}")
print(f"  Total gesture segments: {total_gestures}")
print(f"  Labeled files saved to: {LABELED_DIR}")

Processing gesture files...

horizontal_1.csv
  Gesture: horizontal
  Segments labeled: 23
  Total samples: 3,486
    horizontal:  1,401 samples ( 40.2%)
    dontcare  :  2,085 samples ( 59.8%)

horizontal_10.csv
  Gesture: horizontal
  Segments labeled: 21
  Total samples: 4,293
    horizontal:  1,136 samples ( 26.5%)
    dontcare  :  3,157 samples ( 73.5%)

horizontal_11.csv
  Gesture: horizontal
  Segments labeled: 22
  Total samples: 6,660
    horizontal:  2,487 samples ( 37.3%)
    dontcare  :  4,173 samples ( 62.7%)

horizontal_12.csv
  Gesture: horizontal
  Segments labeled: 21
  Total samples: 3,146
    horizontal:    757 samples ( 24.1%)
    dontcare  :  2,389 samples ( 75.9%)

horizontal_13.csv
  Gesture: horizontal
  Segments labeled: 21
  Total samples: 7,118
    horizontal:  1,136 samples ( 16.0%)
    dontcare  :  5,982 samples ( 84.0%)

horizontal_2.csv
  Gesture: horizontal
  Segments labeled: 24
  Total samples: 6,532
    horizontal:  1,192 samples ( 18.2%)
    dontcare

### 2.4 Verify Labeled Data

Let's verify the labeled data by checking one file.

In [8]:
# Check one labeled file
if labeled_files:
    sample_labeled = pd.read_csv(labeled_files[0])

    print(f"Sample labeled file: {os.path.basename(labeled_files[0])}")
    print(f"Columns: {list(sample_labeled.columns)}")
    print(f"\nLabel distribution:")

    for label_id, count in sample_labeled['label'].value_counts().sort_index().items():
        label_name = LABEL_NAMES[label_id]
        print(f"  {label_name}: {count:,} samples")

    print(f"\nFirst few rows:")
    print(sample_labeled[sample_labeled.columns].head(10))

Sample labeled file: horizontal_labeled_horizontal_1.csv
Columns: ['AccelerometerX', 'AccelerometerY', 'AccelerometerZ', 'GyroscopeX', 'GyroscopeY', 'GyroscopeZ', 'Trigger', 'label']

Label distribution:
  horizontal: 1,401 samples
  dontcare: 2,085 samples

First few rows:
   AccelerometerX  AccelerometerY  AccelerometerZ  GyroscopeX  GyroscopeY  \
0            2002            -457             111         -16          35   
1            1993            -456             116         -17          26   
2            1976            -456             112         -16          -6   
3            1970            -461             112         -12          -7   
4            1966            -465             123          -7           8   
5            1981            -464             122           1          21   
6            1995            -464             125           6          34   
7            1996            -464             123          12          41   
8            1989            -46

## Section 3: Create MPLAB ML Project Files

Now we'll create the `.dclproj` manifest file that MPLAB ML uses to organize labeled data. This file describes:
- Which CSV files contain your data
- How the data is segmented (labeled regions)
- Metadata to set dataset type as test/ train

The .dclproj format allows MPLAB ML to understand the structure of your labeled data without having to manually label using the GUI.

### 3.1 Extract Labeled Segments

For the .dclproj file, we need to identify contiguous regions of each label. Let us define a helper function for it.

In [ ]:
def extract_segments_for_dclproj(df):
    """
    Extract contiguous labeled segments for .dclproj format.

    Returns list of segments:
    [
      {"start": 0, "end": 100, "name": "Label", "value": "horizontal"},
      {"start": 101, "end": 200, "name": "Label", "value": "unknown"},
      ...
    ]
    """
    segments = []

    if 'label' not in df.columns:
        return segments

    # Find where label changes
    label_changes = df['label'].ne(df['label'].shift()).cumsum()

    for group_id in label_changes.unique():
        group_mask = label_changes == group_id
        group_indices = df[group_mask].index

        start_idx = int(group_indices[0])
        end_idx = int(group_indices[-1])
        label_id = int(df.loc[start_idx, 'label'])
        label_name = LABEL_NAMES.get(label_id, 'unknown')

        segments.append({
            "name": "Label",
            "value": label_name,
            "start": start_idx,
            "end": end_idx,
        })

    return segments

print("Segment extraction function defined")

Segment extraction function defined


Here we define another helper function to check if the CSV filename has a _t in it. Such data will be set with metadata "test". Otherwise it will be set to "train" dataset type for metadata.

In [ ]:
# Helper function to check if this is test/ train data to set the metadata accordingly
def detect_dataset_type(filename):
    """
    Detect dataset type from filename.

    horizontal_1.csv   -> training
    horizontal_t1.csv  -> testing
    """
    basename = os.path.basename(filename).lower()

    if "_t" in basename:
        return "testing"
    else:
        return "training"

### 3.2 Create .dclproj Manifest

The manifest file lists all CSV files and their labeled segments.

In [ ]:
def create_dclproj_manifest(labeled_csv_files, output_path):
    """
    Create .dclproj manifest file for MPLAB ML.

    Args:
        labeled_csv_files: List of paths to labeled CSV files
        output_path: Where to save the .dclproj file
    """
    manifest = []

    print("Creating manifest...")
    print("="*70)

    for csv_file in labeled_csv_files:
        basename = os.path.basename(csv_file)
        print(f"  Processing: {basename}")
        # Detect dataset type from filename
        dataset_type = detect_dataset_type(basename)
        # Load labeled data
        df = pd.read_csv(csv_file)

        # Extract segments
        segments = extract_segments_for_dclproj(df)
        print(f"    Found {len(segments)} segments")

        # Build manifest entry
        manifest.append({
            "file_name": basename,
            "metadata": [
                {"name": "dataset_type", "value": dataset_type},
            ],
            "sessions": [
                {
                    "session_name": "Session 1",
                    "segments": segments
                }
            ]
        })

    # Write manifest to file
    with open(output_path, 'w') as f:
        json.dump(manifest, f, indent=2)

    print(f"\n{'='*70}")
    print(f"Manifest created: {output_path}")
    print(f"  Total files: {len(manifest)}")
    total_segments = sum(len(item['sessions'][0]['segments']) for item in manifest)
    print(f"  Total segments: {total_segments}")

    return output_path

print("Manifest creation function defined")

Manifest creation function defined


### 3.3 Generate Project Files

Let's create the .dclproj file and organize everything for upload.

In [ ]:
# Configuration

#The project name is the root folder name obtained froms ection 1.3
PROJECT_NAME = folder_name
print("PROJECT_NAME:", PROJECT_NAME)
# Create MPLAB project directory
MPLAB_PROJECT_DIR = '/content/mplab_project'
os.makedirs(MPLAB_PROJECT_DIR, exist_ok=True)

# Copy labeled CSV files to project directory
print("Preparing MPLAB ML project files...")
print("="*70)

import shutil

mplab_csv_files = []
for labeled_file in labeled_files:
    basename = os.path.basename(labeled_file)
    dest_path = os.path.join(MPLAB_PROJECT_DIR, basename)
    shutil.copy2(labeled_file, dest_path)
    mplab_csv_files.append(dest_path)
    print(f"  Copied: {basename}")

# Create .dclproj manifest
manifest_filename = f"{PROJECT_NAME.lower()}.dclproj"
manifest_path = os.path.join(MPLAB_PROJECT_DIR, manifest_filename)

create_dclproj_manifest(
    mplab_csv_files,
    manifest_path)

print(f"\n{'='*70}")
print(f"MPLAB ML project ready!")
print(f"  Location: {MPLAB_PROJECT_DIR}")
print(f"  Manifest: {manifest_filename}")
print(f"  CSV files: {len(mplab_csv_files)}")

PROJECT_NAME: MagicWandGame
Preparing MPLAB ML project files...
  Copied: 4_labeled_4_1.csv
  Copied: 4_labeled_4_10.csv
  Copied: 4_labeled_4_11.csv
  Copied: 4_labeled_4_12.csv
  Copied: 4_labeled_4_13.csv
  Copied: 4_labeled_4_14.csv
  Copied: 4_labeled_4_2.csv
  Copied: 4_labeled_4_3.csv
  Copied: 4_labeled_4_4.csv
  Copied: 4_labeled_4_5.csv
  Copied: 4_labeled_4_6.csv
  Copied: 4_labeled_4_7.csv
  Copied: 4_labeled_4_8.csv
  Copied: 4_labeled_4_9.csv
  Copied: 4_labeled_4_t1.csv
  Copied: 4_labeled_4_t2.csv
  Copied: 4_labeled_4_t3.csv
  Copied: 4_labeled_4_t4.csv
  Copied: 4_labeled_4_t5.csv
  Copied: 4_labeled_4_t6.csv
  Copied: 4_labeled_4_t7.csv
  Copied: b_labeled_b_1.csv
  Copied: b_labeled_b_10.csv
  Copied: b_labeled_b_11.csv
  Copied: b_labeled_b_12.csv
  Copied: b_labeled_b_13.csv
  Copied: b_labeled_b_14.csv
  Copied: b_labeled_b_15.csv
  Copied: b_labeled_b_16.csv
  Copied: b_labeled_b_17.csv
  Copied: b_labeled_b_18.csv
  Copied: b_labeled_b_2.csv
  Copied: b_labeled

## Section 4: Upload to MPLAB ML
We are now ready to upload this project to MPLAB ML Developer Suite



### 4.1 Install MPLAB ML SDK
We'll install the SDK to use the APIs to connect to the cloud platform and upload the project.

In [ ]:
# Install MPLAB ML SDK for project file creation
print("Installing MPLAB ML Development Suite SDK...")
!pip install mplabml --quiet

# Verify installation
try:
    import mplabml
    from mplabml.dclproj import dclproj
    print(f"✓ MPLAB ML SDK installed successfully")
    print(f"  Version: {mplabml.__version__}")
    print("\nReady to create MPLAB ML project files")
except ImportError as e:
    print(f"Error: {e}")
    print("Installation may have failed - please restart runtime and try again")

Installing MPLAB ML Development Suite SDK...


✓ MPLAB ML SDK installed successfully
  Version: 2025.1.1

Ready to create MPLAB ML project files


You will be prompted to enter the API key generated in your MPLAB ML Development Suite account.

In [ ]:
from mplabml import Client
from IPython.display import display, Javascript
# Force visible input prompt
api_key = input("🔑 Enter your MPLAB ML API Key: ").strip()
if not api_key:
    raise ValueError("API key is required!")

client = Client(api_key=api_key)
display(Javascript("alert('Connected to MPLAB ML Cloud successfully!');"))

🔑 Enter your MPLAB ML API Key: oTz40Oih.x6HElxOmJ7DMCO0gpq4iiMXhq2rGiDGG
Client connected to the cloud successfully


<IPython.core.display.Javascript object>

### 4.2 Upload the project to the cloud platform
We will now use the SDK API upload_project_dcli to upload the project manifest file we created in section 4.3.

In [ ]:
OUTPUT_DIR = MPLAB_PROJECT_DIR
print("=" * 70)
print(f"UPLOADING PROJECT: {PROJECT_NAME}")
print("=" * 70)

print(f"\nManifest: {manifest_path}")
print(f"Upload directory: {OUTPUT_DIR}")

# Verify manifest exists
if not os.path.exists(manifest_path):
    raise FileNotFoundError(f"Manifest not found at: {manifest_path}")

# -------------------------------------------------
# Check if project exists and delete it
# -------------------------------------------------

print("\n🔍 Checking for existing project...")

try:
    projects_df = client.list_projects()

    if PROJECT_NAME in projects_df['Name'].values:
        print(f"⚠️ Project '{PROJECT_NAME}' already exists")
        print("🗑️ Deleting existing project...")
        client.delete_project(PROJECT_NAME)
        print("✓ Deleted successfully")
    else:
        print("✓ No existing project found")

except Exception as e:
    print(f"ℹ️ Could not check for existing project: {e}")

# -------------------------------------------------
# Upload Project
# -------------------------------------------------

print("\n⏳ Upload in progress...")
print("This may take 30-60 seconds...\n")

try:
    result = client.upload_project_dcli(
        name=PROJECT_NAME,
        dcli_path=manifest_path,
        force=True
    )

    print("\n" + "=" * 70)
    print("✅ UPLOAD COMPLETED SUCCESSFULLY!")
    print("=" * 70)
    print(f"\nProject '{PROJECT_NAME}' is now in MPLAB ML!\n")
    display(Javascript("alert(' Notebook executed successfully and your project is now uploaded!');"))
except Exception as e:
    print("\n" + "=" * 70)
    print("❌ UPLOAD FAILED")
    print("=" * 70)
    print(f"\nError: {e}\n")
    raise

UPLOADING PROJECT: MagicWandGame

Manifest: /content/mplab_project/magicwandgame.dclproj
Upload directory: /content/mplab_project

🔍 Checking for existing project...
✓ No existing project found

⏳ Upload in progress...
This may take 30-60 seconds...

Project MagicWandGame does not exist, creating a new project.
Uploading Capture 0/128: 4_labeled_4_1.csv
Segmenter Uploaded.
Uploading Capture 1/128: 4_labeled_4_10.csv
Uploading Capture 2/128: 4_labeled_4_11.csv
Uploading Capture 3/128: 4_labeled_4_12.csv
Uploading Capture 4/128: 4_labeled_4_13.csv
Uploading Capture 5/128: 4_labeled_4_14.csv
Uploading Capture 6/128: 4_labeled_4_2.csv
Uploading Capture 7/128: 4_labeled_4_3.csv
Uploading Capture 8/128: 4_labeled_4_4.csv
Uploading Capture 9/128: 4_labeled_4_5.csv
Uploading Capture 10/128: 4_labeled_4_6.csv
Uploading Capture 11/128: 4_labeled_4_7.csv
Uploading Capture 12/128: 4_labeled_4_8.csv
Uploading Capture 13/128: 4_labeled_4_9.csv
Uploading Capture 14/128: 4_labeled_4_t1.csv
Uploading C

With this, you will have created a project in MPLAB ML Developer Suite cloud platform and uploaded all the collected and labelled data to it without having to manually add labels or metadata.